AIFM PE Buyout FundThis notebook presents a private equity risk-monitoring workflow for a simulated AIF portfolio. The fund represents a European mid-market buyout strategy with portfolio companies across technology, healthcare, industrials, consumer, and energy transition sectors.The analysis focuses on private-capital indicators, including IRR, MOIC, DPI, RVPI, PME, unfunded commitments, capital calls, valuation bridge, sector exposure, and selected sustainability indicators. Regulatory context is AIFMD-style fund governance, but the notebook focuses on the monitoring workflow and interpretation of the resulting risk metrics.The simulated fund is `AIFM_PE_Buyout`, a 2018-vintage buyout fund with a EUR 200m target size and 10-year fund life.> **Output gallery:** Tables and plots generated by this notebook are saved in the [`figs/aifm_pe`](../../figs/aifm_pe) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.> For supporting methodology notes, see [PE Buyout Notes](../../docs/notebooks_notes/pe_buyout.md).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# to initialize db if no tables exist abd to create session factory
from src.setup_db import run as setup_db
setup_db()
from src.data.database import get_engine
ENGINE = get_engine()

# initialize mock bloomberg API
from src.data.mock_bloomberg import MockBloomberg as Bloomberg
BBG = Bloomberg()

# helper functions for risk computations and nice output formatting
import src.risk.risk_utils as rk
import src.ui.print_html_utils as phtml

### Fund Example in this Notebook

The fund profile below sets the operating context for the risk workflow. It defines the strategy, fund type, base currency, reporting setup, and monitoring framework used by the calculations that follow.

<small>

In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID = 'AIFM_PE_Buyout'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated. They are used to show how a fund-level risk framework can be represented in a structured workflow.

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="02",
)

The RMP is loaded as `rmp` and passed to the relevant risk functions. This keeps fund-specific parameters outside the calculation code.

In [ ]:
# read in RMP parameters for the fund
from src.data.reference_data import load_rmp
rmp = load_rmp(FUND_ID)

### Implementation Context

The analysis is performed as of a fixed valuation date, consistent with the point-in-time design used across the fund workflows.

In [ ]:
# fixed valuation date for all computations in this notebook
from src.config import VALUATION_DATE
VALUATION_DATE

Portfolio positions, fund characteristics, counterparty records, reference data, and market data are retrieved from the SQLite data layer. Market data is enriched through the simulated Bloomberg workflow before being passed to the risk analytics modules.

For a fuller explanation of the data workflow, see the [Data Layer Workflow](../data_workflows/01_data_layer_workflow.ipynb).

In [ ]:
# enrich fund postion data w/ info from Bloomberg
from src.data.enrichment import get_risk_ready_df
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# display the enriched risk dataframe first 2 rows
risk_df.head(2)

From this point onward, the notebook uses helper functions to render summary tables and HTML views. Data aggregation, calculations, and reporting logic remain inside the package modules, so the notebook stays focused on review and interpretation.

In [ ]:
# query from SQLdb
from src.data.database import query_positions
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)
NAV = risk_df['market_value_eur'].sum()

phtml.display_fund_summary(FUND_ID, VALUATION_DATE, positions, risk_df, NAV, export_id="03")

In [ ]:
phtml.display_top_positions(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="04")

In [ ]:
phtml.display_asset_class_breakdown(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="05")  # Also computes NAV internally

---

### Risk Management Policy Parameters

The fund's risk parameters are sourced from the Risk Management Policy configuration and used throughout the notebook for measurement, monitoring, and limit checks.

In [ ]:
# from src.config import VALUATION_DATE, QUARTER
# from src.risk.reg_constants import CONFIDENCE, HORIZON, NOTICE, GROSS_LIMIT

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import warnings
# warnings.filterwarnings('ignore')

# from src.risk.pe_utils import pe_value_bridge
# from src.setup_db import run as setup_db
# setup_db()

# from src.ui.plot_style import (
#     C,  FUND_COLORS, ACCENT, ACCENT2, ACCENT3, WARNING,
#     apply_ax_style, section_title, fig_header, fig_footer,
#     callout_box, threshold_vline, threshold_hline, breach_fill,
#     pct_color, rag_color, util_color, liq_color, table_cell, sup_title,
# )
# from src.ui.nb_utils import save_fig, save_table, styled_table, save_table_png


# from src.data.database import (
#     get_engine, PEFund, PEPortfolioCompany,
#     PEFundInvestment, PECashFlow, PENavHistory
# )
# from src.data.mock_bloomberg import MockBloomberg as Bloomberg
# import src.risk.risk_utils as rk
# import src.ui.print_html_utils as phtml
# import src.print_utils as prt
# import src.risk.esg_utils as esg_u 
# from src.risk.esg_utils import ESG_FIELDS, ESG_THRESHOLD_LOW
# from src.ui.nb_utils import save_fig, save_table, styled_table
# from sqlalchemy.orm import Session

# from src.data.generate_pe_fund import generate_cash_flows, COMPANIES, COMMITTED
# from src.risk.pe_utils import pe_value_bridge

# flows = generate_cash_flows()

# FUND_ID = 'AIFM_PE_Buyout'
# VALUATION_DATE   = '2026-05-13'
# ENGINE  = get_engine()
# BBG     = Bloomberg()

In [ ]:
# Display fund overview banner — fund identity and regulatory classification
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
)

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
)

In [ ]:
len('fund_structure.committed_capital_eur')


1. Load and Validate Fund DataPE fund data is organised around portfolio companies, cash flows, and quarterly NAV appraisals rather than daily position snapshots.The notebook loads dedicated PE tables from SQLite:* `pe_funds`* `pe_portfolio_companies`* `pe_fund_investments`* `pe_cash_flows`* `pe_nav_history`

In [ ]:
# MRS-65: Load and validate PE fund data
with Session(ENGINE) as session:
    fund        = session.query(PEFund).filter_by(fund_id=FUND_ID).first()
    companies   = session.query(PEPortfolioCompany).all()
    investments = session.query(PEFundInvestment).filter_by(fund_id=FUND_ID).all()
    cash_flows  = session.query(PECashFlow).filter_by(fund_id=FUND_ID).all()
    nav_history = session.query(PENavHistory).filter_by(fund_id=FUND_ID).all()

cf_df  = pd.DataFrame([{
    'date'       : cf.date,
    'company_id' : cf.company_id,
    'flow_type'  : cf.flow_type,
    'amount_eur' : cf.amount_eur,
    'description': cf.description,
} for cf in cash_flows])
cf_df['date'] = pd.to_datetime(cf_df['date'])

nav_df = pd.DataFrame([{
    'date'           : n.date,
    'company_id'     : n.company_id,
    'nav_eur'        : n.nav_eur,
    'gross_multiple' : n.gross_multiple,
    'unrealised_gain': n.unrealised_gain,
    'cost_basis_eur' : n.cost_basis_eur,
} for n in nav_history])
nav_df['date'] = pd.to_datetime(nav_df['date'])

inv_df = pd.DataFrame([{
    'company_id'     : i.company_id,
    'investment_date': i.investment_date,
    'cost_basis_eur' : i.cost_basis_eur,
    'ownership_pct'  : i.ownership_pct,
    'entry_ev_ebitda': i.entry_ev_ebitda,
    'entry_ev_sales' : i.entry_ev_sales,
    'exit_date'      : i.exit_date,
} for i in investments])

co_df = pd.DataFrame([{
    'company_id'      : c.company_id,
    'company_name'    : c.company_name,
    'sector'          : c.sector,
    'country'         : c.country,
    'investment_stage': c.investment_stage,
    'status'          : c.status,
} for c in companies])

call_flows = [f for f in flows if f['flow_type'] == 'capital_call']
fee_flows  = [f for f in flows if f['flow_type'] == 'management_fee']

total_called = sum(abs(f['amount_eur']) for f in call_flows)
total_fees   = sum(abs(f['amount_eur']) for f in fee_flows)
total_dist   = sum(f['amount_eur'] for f in flows 
                   if f['flow_type'] in ('distribution', 'exit_proceeds'))
gp_carry     = sum(f['amount_eur'] for f in flows 
                   if f['flow_type'] == 'carried_interest')


with ENGINE.connect() as conn:
    df_latest_nav = pd.read_sql(
        """SELECT company_id, nav_eur 
           FROM pe_nav_history
           WHERE fund_id = 'AIFM_PE_Buyout'
           AND company_id IS NOT NULL
           AND date = (
               SELECT MAX(date) FROM pe_nav_history nh2
               WHERE nh2.fund_id = pe_nav_history.fund_id
               AND nh2.company_id = pe_nav_history.company_id
           )""",
        conn
    )

# only active companies -- exited ones have no current NAV
active_ids = [c['company_id'] for c in COMPANIES if not c.get('exit_date')]
current_nav = df_latest_nav[df_latest_nav['company_id'].isin(active_ids)]['nav_eur'].sum()

print(f"Total capital called  : EUR {total_called:,.0f}  ({total_called/COMMITTED*100:.1f}% of committed)")
print(f"Total management fees : EUR {total_fees:,.0f}")
print(f"Total LP distributed  : EUR {total_dist:,.0f}")
print(f"GP carried interest   : EUR {gp_carry:,.0f}")
print(f"Current NAV (active companies only): EUR {current_nav:,.0f}")


In [ ]:
from src.data.generate_pe_fund import generate_valuation_reports, COMPANIES as COMP_LIST, compute_entry_equity_check

val_reports = generate_valuation_reports()

# ── build portfolio dataframe first ─────────────────────────────────────────
portfolio = co_df.merge(inv_df, on='company_id')
portfolio['investment_date'] = pd.to_datetime(portfolio['investment_date'])
portfolio['exit_date']       = pd.to_datetime(portfolio['exit_date'])

# ── derive exit multiple from appraisal NAV at exit date ────────────────────
def get_exit_multiple(row):
    if pd.isna(row['exit_date']):
        return None
    exit_date = row['exit_date'].strftime('%Y-%m-%d')
    cid = row['company_id']
    company_reports = [
        r for r in val_reports
        if r['company_id'] == cid and r['date'] <= exit_date
    ]
    if not company_reports:
        return None
    latest_nav = max(company_reports, key=lambda r: r['date'])['appraised_nav_eur']
    cost_basis = compute_entry_equity_check(cid)
    return latest_nav / cost_basis

portfolio['exit_multiple'] = portfolio.apply(get_exit_multiple, axis=1)

# ── display ──────────────────────────────────────────────────────────────────
portfolio_display = portfolio[[
    'company_name', 'sector', 'country', 'investment_stage',
    'status', 'investment_date', 'cost_basis_eur',
    'ownership_pct', 'entry_ev_ebitda', 'entry_ev_sales', 'exit_date', 'exit_multiple'
]].copy()

portfolio_display['investment_date'] = portfolio_display['investment_date'].dt.strftime('%Y-%m-%d')
portfolio_display['exit_date']       = portfolio_display['exit_date'].dt.strftime('%Y-%m-%d').fillna('--')
portfolio_display['exit_multiple']   = portfolio_display['exit_multiple'].map(
    lambda x: f'{x:.2f}x' if pd.notna(x) else '--')
portfolio_display['cost_basis_eur']  = portfolio_display['cost_basis_eur'].map('{:,.0f}'.format)
portfolio_display['ownership_pct']   = portfolio_display['ownership_pct'].map('{:.1f}%'.format)
portfolio_display['entry_ev_ebitda'] = portfolio_display['entry_ev_ebitda'].map(
    lambda x: f'{x:.1f}x' if pd.notna(x) else '--')
portfolio_display['entry_ev_sales']  = portfolio_display['entry_ev_sales'].map(
    lambda x: f'{x:.1f}x' if pd.notna(x) else '--')
portfolio_display.columns = [
    'Company', 'Sector', 'Country', 'Stage',
    'Status', 'Invested', 'Cost (EUR)',
    'Ownership', 'Entry EV/EBITDA', 'Entry EV/Sales', 'Exit Date', 'Exit Multiple'
]
portfolio_display.set_index('Company', inplace=True)
portfolio_display

## 2. Independent Appraisal and Valuation ReportsPE portfolio companies are valued quarterly using independent appraisal inputs stored in `pe_valuation_report`.The notebook uses the valuation report to review appraised NAV, LTM financials, EV/EBITDA multiples, net debt, leverage, covenant status, and key risks.Valuation risk is treated as model and appraisal risk rather than daily market-price risk.

In [ ]:
from src.data.database import PEValuationReport

with Session(ENGINE) as session:
    val_reports = session.query(PEValuationReport).filter_by(fund_id=FUND_ID).all()

vr_df = pd.DataFrame([{
    'company_id'        : v.company_id,
    'date'              : v.date,
    'appraised_nav_eur' : v.appraised_nav_eur,
    'ebitda_ltm_eur'    : v.ebitda_ltm_eur,
    'revenue_ltm_eur'   : v.revenue_ltm_eur,
    'ebitda_margin'     : v.ebitda_margin,
    'net_debt_eur'      : v.net_debt_eur,
    'ev_eur'            : v.ev_eur,
    'ev_ebitda'         : v.ev_ebitda,
    'leverage_ratio'    : v.leverage_ratio,
    'leverage_covenant' : v.leverage_covenant,
    'coverage_ratio'    : v.coverage_ratio,
    'coverage_covenant' : v.coverage_covenant,
    'covenant_type'     : v.covenant_type,
    'appraiser'         : v.appraiser,
    'key_risks'         : v.key_risks,
    'arr_eur'           : v.arr_eur,
} for v in val_reports])
vr_df['date'] = pd.to_datetime(vr_df['date'])

print(f"Valuation reports loaded : {len(vr_df)}")
print(f"Companies covered        : {vr_df['company_id'].nunique()}")
print(f"Date range               : {vr_df['date'].min().date()} to {vr_df['date'].max().date()}")
print()

def format_leverage(row):
    if pd.isna(row['leverage_ratio']) or row['leverage_ratio'] > 50:
        return 'N/A (neg. EBITDA)'
    return f"{row['leverage_ratio']:.2f}x"

def covenant_headroom(row):
    if pd.isna(row['leverage_covenant']) or pd.isna(row['leverage_ratio']):
        return '—'
    if row['leverage_ratio'] > 50:
        return 'BREACH'
    headroom = (row['leverage_covenant'] - row['leverage_ratio']) / row['leverage_covenant'] * 100
    if headroom < 0:
        return f'BREACH ({headroom:.1f}%)'
    elif headroom < 20:
        return f'⚠ {headroom:.1f}%'
    return f'{headroom:.1f}%'

latest = vr_df.sort_values('date').groupby('company_id').last().reset_index()
latest['leverage_display'] = latest.apply(format_leverage, axis=1)
latest['headroom']         = latest.apply(covenant_headroom, axis=1)

print(latest[['company_id', 'appraised_nav_eur', 'leverage_display', 
              'leverage_covenant', 'headroom', 'covenant_type']].to_string())

## 3. J-Curve AnalysisThe J-curve tracks cumulative net cash flow over the fund life. Capital calls create negative cash flow early in the fund, while distributions from exits and recapitalisations can move cumulative net cash flow back toward positive territory.```textNet cash flow = distributions - capital calledCumulative net cash flow = cumulative distributions - cumulative capital called```

In [ ]:
# MRS-55: J-curve visualisation

import matplotlib.ticker as mticker
from sqlalchemy import text

# ── 1. Pull and aggregate cash flows by quarter ──────────────────────────────

with ENGINE.connect() as conn:
    df_cf = pd.read_sql(
        "SELECT date, flow_type, amount_eur FROM pe_cash_flows WHERE fund_id = 'AIFM_PE_Buyout'",
        conn
    )
    df_nav_raw = pd.read_sql(
        "SELECT company_id, date, nav_eur FROM pe_nav_history WHERE fund_id = 'AIFM_PE_Buyout'",
        conn
    )

df_cf["date"] = pd.to_datetime(df_cf["date"])
df_nav_raw["date"] = pd.to_datetime(df_nav_raw["date"])

# quarter-end label for grouping
df_cf["quarter"] = df_cf["date"].dt.to_period("Q").dt.to_timestamp("Q")
df_nav_raw["quarter"] = df_nav_raw["date"].dt.to_period("Q").dt.to_timestamp("Q")

# capital calls: amount_eur is negative in DB -- flip to positive for display
capital_called = (
    df_cf[df_cf["flow_type"] == "capital_call"]
    .groupby("quarter")["amount_eur"]
    .sum()
    .abs()
    .rename("capital_called")
)

# distributions: positive amounts
distributions = (
    df_cf[df_cf["flow_type"].isin(["distribution", "exit_proceeds"])]
    .groupby("quarter")["amount_eur"]
    .sum()
    .rename("distributions")
)

# management fees included in net cash flow but not in capital called
mgmt_fees = (
    df_cf[df_cf["flow_type"] == "management_fee"]
    .groupby("quarter")["amount_eur"]
    .sum()
    .abs()
    .rename("mgmt_fees")
)

# fund-level NAV = sum of company-level NAVs per quarter (exclude fund aggregate row)
nav_by_quarter = (
    df_nav_raw[df_nav_raw["company_id"].notna()]
    .groupby("quarter")["nav_eur"]
    .sum()
    .rename("nav")
)

# ── 2. Build master quarterly frame ─────────────────────────────────────────

quarters = nav_by_quarter.index.union(capital_called.index).union(distributions.index)

cf = pd.DataFrame(index=quarters)
cf = cf.join(capital_called).join(distributions).join(mgmt_fees).join(nav_by_quarter)
cf = cf.fillna(0)
cf = cf.sort_index()
cf.index.name = "quarter"

# net cash flow: distributions out minus capital in minus fees
cf["ncf"]      = cf["distributions"] - cf["capital_called"] - cf["mgmt_fees"]
cf["cncf"]     = cf["ncf"].cumsum()
cf["pic"]      = cf["capital_called"].cumsum()
cf["cum_dist"] = cf["distributions"].cumsum()

cf["dpi"]  = cf["cum_dist"] / cf["pic"].replace(0, float("nan"))
cf["rvpi"] = cf["nav"]      / cf["pic"].replace(0, float("nan"))
cf["tvpi"] = cf["dpi"] + cf["rvpi"]

trough_idx  = cf["cncf"].idxmin()
trough_val  = cf.loc[trough_idx, "cncf"]

print(f"Quarters in dataset : {len(cf)}")
print(f"Trough quarter      : {trough_idx.strftime('%Q-%Y')}")
print(f"Trough CNCF         : {trough_val/1e6:,.1f}M EUR")
print(f"\nLatest multiples (as of {cf.index[-1].strftime('%Q-%Y')}):")
print(f"  DPI  : {cf['dpi'].iloc[-1]:.2f}x")
print(f"  RVPI : {cf['rvpi'].iloc[-1]:.2f}x")
print(f"  TVPI : {cf['tvpi'].iloc[-1]:.2f}x")

# ── 3. Plot ──────────────────────────────────────────────────────────────────
trough_idx = cf['cncf'].idxmin()
trough_val = cf.loc[trough_idx, 'cncf']
dates      = cf.index.to_timestamp() if hasattr(cf.index, 'to_timestamp') else cf.index

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 9),
    gridspec_kw={'height_ratios': [2, 1]},
    sharex=True
)

ax1.plot(dates, cf['nav'], color=ACCENT2, linewidth=1.4,
         linestyle=':', label='NAV (unrealised)')
ax1.bar(dates, -cf['capital_called'],
        width=60, color=C['red'], alpha=0.75, label='Capital called')
ax1.bar(dates, -cf['mgmt_fees'],
        width=60, color=C['amber'], alpha=0.65,
        bottom=-cf['capital_called'], label='Management fees')
ax1.bar(dates, cf['distributions'],
        width=60, color=C['green'], alpha=0.75, label='Distributions')
ax1.plot(dates, cf['cncf'], color='white', linewidth=2.2,
         marker='o', markersize=3.5, label='Cumulative NCF')
ax1.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax1.annotate(
    f'Trough\n{trough_idx.strftime("%b %Y")}\n{trough_val/1e6:.1f}M',
    xy=(trough_idx, trough_val),
    xytext=(trough_idx, trough_val * 0.6),
    arrowprops=dict(arrowstyle='->', color=C['red']),
    fontsize=8, color=C['red'], ha='center'
)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax1.set_ylabel('EUR')
section_title(ax1, 'J-Curve — AIFM PE Buyout Fund', fontsize=14)
ax1.legend(loc='upper left', fontsize=8)

ax2.plot(dates, cf['tvpi'], color='white',  linewidth=2,   label='TVPI')
ax2.plot(dates, cf['dpi'],  color=C['green'], linewidth=1.6, linestyle='--', label='DPI')
ax2.plot(dates, cf['rvpi'], color=ACCENT,   linewidth=1.6, linestyle=':',  label='RVPI')
ax2.axhline(1.0, color='grey', linewidth=0.8, linestyle='--')
ax2.set_ylabel('Multiple (x)')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(False)
ax2.spines[['top', 'right']].set_visible(False)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}x'))

plt.tight_layout()
save_fig(fig, FUND_ID, '07. PE j-curve')
plt.show()

In [ ]:
print(cf.columns.tolist())
print(cf.head())

4. Exit WaterfallThe exit waterfall shows how proceeds from a portfolio company sale are allocated between LPs and the GP.The simulated waterfall follows a European-style sequence: return of capital, preferred return, GP catch-up, and carried-interest split.

In [ ]:
from src.data.generate_pe_fund import (
    generate_cash_flows, generate_valuation_reports,
    COMPANIES, HURDLE_RATE, CARRY_RATE
)
import matplotlib.patches as mpatches

val_reports = generate_valuation_reports()
flows       = generate_cash_flows(val_reports)
call_flows  = [f for f in flows if f['flow_type'] == 'capital_call']
fee_flows   = [f for f in flows if f['flow_type'] == 'management_fee']

exit_companies = [c for c in COMPANIES if c.get('exit_date')]

LP_COLOR = C['blue2']
GP_COLOR = C['green']

fig, axes = plt.subplots(1, len(exit_companies), figsize=(16, 8), sharey=False)

for ax, ex in zip(axes, exit_companies):
    cid       = ex['company_id']
    exit_date = ex['exit_date']
    name      = ex['company_name'].replace(' ', '\n')

    company_reports = [r for r in val_reports
                       if r['company_id'] == cid and r['date'] <= exit_date]
    gross = max(company_reports, key=lambda r: r['date'])['appraised_nav_eur']

    contributed = sum(abs(f['amount_eur']) for f in call_flows if f['company_id'] == cid)
    n_active    = len([c for c in COMPANIES
                       if pd.Timestamp(c['investment_date']) <= pd.Timestamp(exit_date)])
    fees_paid   = sum(abs(f['amount_eur']) / n_active for f in fee_flows
                      if f['date'] <= exit_date)
    total_contributed = contributed + fees_paid

    first_call = min(f['date'] for f in call_flows if f['company_id'] == cid)
    years      = (pd.Timestamp(exit_date) - pd.Timestamp(first_call)).days / 365
    preferred_return = total_contributed * ((1 + HURDLE_RATE) ** years - 1)

    remaining = gross
    steps     = []

    roc = min(remaining, total_contributed)
    steps.append(('Return of\nCapital',   roc,    LP_COLOR, 'LP'))
    remaining -= roc

    if remaining > 0:
        pref = min(remaining, preferred_return)
        steps.append(('Preferred\nReturn 8%', pref, LP_COLOR, 'LP'))
        remaining -= pref

    if remaining > 0:
        total_profit = gross - total_contributed
        gp_target    = total_profit * CARRY_RATE
        catchup      = min(remaining, gp_target)
        steps.append(('GP\nCatch-up', catchup, GP_COLOR, 'GP'))
        remaining -= catchup

    if remaining > 0:
        steps.append(('LP 80%\nSplit', remaining * (1 - CARRY_RATE), LP_COLOR, 'LP'))
        steps.append(('GP 20%\nSplit', remaining * CARRY_RATE,        GP_COLOR, 'GP'))

    bottom = 0
    bar_labels, bar_bottoms, bar_heights, bar_colors = [], [], [], []
    for label, amount, color, party in steps:
        bar_labels.append(label)
        bar_bottoms.append(bottom)
        bar_heights.append(amount)
        bar_colors.append(color)
        bottom += amount

    for i, (h, b, c) in enumerate(zip(bar_heights, bar_bottoms, bar_colors)):
        ax.bar(i, h, bottom=b, color=c, alpha=0.85, width=0.6)
        ax.text(i, b + h / 2, f'{h/1e6:.1f}M',
                ha='center', va='center', fontsize=10, color='white', fontweight='bold')

    ax.axhline(gross / 1e6, color='white', linewidth=1.2, linestyle='--', alpha=0.6)
    ax.text(len(bar_labels) - 0.4, gross / 1e6,
            f'Gross\n{gross/1e6:.1f}M', fontsize=10, va='bottom', color='white')
    ax.set_xticks(list(range(len(bar_labels))))
    ax.set_xticklabels(bar_labels, fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.0f}M'))
    ax.tick_params(axis='y', labelsize=10)
    # section_title(ax, f'{name} | {exit_date}', fontsize=11)
    name = ex['company_name'].replace('\n', ' ')
    section_title(ax, f'{name} | {exit_date}', fontsize=13)
 

lp_patch = mpatches.Patch(color=LP_COLOR, alpha=0.85, label='LP')
gp_patch = mpatches.Patch(color=GP_COLOR, alpha=0.85, label='GP')
fig.legend(handles=[lp_patch, gp_patch], loc='upper right', fontsize=11)
sup_title(fig, 'European Waterfall — Exit Proceeds Distribution', fontsize=18)
plt.tight_layout()
save_fig(fig, FUND_ID, '08. european waterfall')
plt.show()

5. Fund Cash Management and Subscription Credit FacilityThe notebook tracks fund-level cash, capital calls, expenses, and use of the subscription credit facility.The subscription line is used as short-term bridge financing backed by unfunded LP commitments. The output shows how cash movements, facility drawdowns, repayments, and interest costs affect fund liquidity.

In [ ]:
# MRS-73: Fund cash management visualisation

with ENGINE.connect() as conn:
    df_cash = pd.read_sql(
        "SELECT date, cash_balance_eur, sub_line_drawn, net_cash_position, "
        "cumulative_interest_earned, cumulative_interest_paid "
        "FROM pe_fund_cash_management WHERE fund_id = 'AIFM_PE_Buyout' ORDER BY date",
        conn
    )

df_cash['date'] = pd.to_datetime(df_cash['date'])

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 11), sharex=True)

# ── panel 1: cash balance vs sub line ───────────────────────────────────────
ax1.bar(df_cash['date'], df_cash['cash_balance_eur'] / 1e6,
        width=60, color=ACCENT, alpha=0.7, label='Cash reserve')
ax1.plot(df_cash['date'], df_cash['sub_line_drawn'] / 1e6,
         color=C['amber'], linewidth=2, marker='o', markersize=4,
         label='Sub line drawn')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax1.set_ylabel('EUR M')
section_title(ax1, 'Fund Cash Management — AIFM PE Buyout', fontsize=13)
ax1.legend(fontsize=8)

# ── panel 2: net cash position ───────────────────────────────────────────────
colors = [C['green'] if v >= 0 else C['red']
          for v in df_cash['net_cash_position']]
ax2.bar(df_cash['date'], df_cash['net_cash_position'] / 1e6,
        width=60, color=colors, alpha=0.8)
ax2.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax2.set_ylabel('EUR M')
section_title(ax2, 'Net Cash Position (Cash reserve - Sub line)', fontsize=10)

# ── panel 3: cumulative interest earned vs paid ──────────────────────────────
ax3.plot(df_cash['date'], df_cash['cumulative_interest_earned'] / 1e6,
         color=C['green'], linewidth=2, label='Cumulative interest earned')
ax3.plot(df_cash['date'], df_cash['cumulative_interest_paid'] / 1e6,
         color=C['red'], linewidth=2, linestyle='--', label='Cumulative interest paid')
ax3.fill_between(df_cash['date'],
                 df_cash['cumulative_interest_earned'] / 1e6,
                 df_cash['cumulative_interest_paid'] / 1e6,
                 alpha=0.15, color=C['green'], label='Net benefit')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}M'))
ax3.set_ylabel('EUR M')
ax3.set_xlabel('Quarter')
section_title(ax3, 'Cumulative Interest: Earned vs Paid', fontsize=10)
ax3.legend(fontsize=8)

plt.tight_layout()
save_fig(fig, FUND_ID, '09. fund cash management')
plt.show()

6. Return Attribution: Value BridgeThe value bridge decomposes equity value creation into operating performance, multiple movement, deleveraging, and FX effects.The output separates operational value creation from valuation and capital-structure effects.

In [ ]:
bridge = pe_value_bridge(ENGINE, FUND_ID)
rows   = bridge['rows']
ft     = bridge['fund_totals']

print(f"Companies in bridge: {len(rows)}")
print(f"  Realised:   {sum(r['is_realised'] for r in rows)}")
print(f"  Unrealised: {sum(not r['is_realised'] for r in rows)}")

In [ ]:
COLOURS = {
    'ebitda_growth':     C['blue2'],
    'multiple_expansion':C['blue2'],
    'leverage_effect':   C['blue2'],
    'distributions':     C['blue2'],
    'gap_material':      '#e74c3c',
    'gap_immaterial':    '#3a3a3a',
    'anchor':            C['cyan'],
}
GAP_THRESHOLD = 0.05

def wrap_label(label: str) -> str:
    words = label.split()
    if len(words) == 2:
        return f'{words[0]}\n{words[1]}'
    return label

def fmt_mm(v: float) -> str:
    return f'{v / 1e6:+.1f}M'

def waterfall_chart(row: dict, ax: plt.Axes) -> None:
    components = [
        ('Entry equity',       row['entry_equity_value'],  COLOURS['anchor']),
        ('EBITDA growth',      row['ebitda_growth'],       COLOURS['ebitda_growth']),
        ('Multiple expansion', row['multiple_expansion'],  COLOURS['multiple_expansion']),
        ('Leverage effect',    row['leverage_effect'],     COLOURS['leverage_effect']),
        ('Distributions',      row['distributions'],       COLOURS['distributions']),
    ]
    gap_pct = row['reconciliation_gap_pct']
    gap_col = (
        COLOURS['gap_material']
        if not np.isnan(gap_pct) and abs(gap_pct) > GAP_THRESHOLD
        else COLOURS['gap_immaterial']
    )
    components.append(('Recon. gap', row['reconciliation_gap'], gap_col))
    running = 0.0
    bottoms, heights, colours, labels = [], [], [], []
    for i, (label, value, colour) in enumerate(components):
        if i == 0:
            bottoms.append(0.0)
            running = value
        else:
            bottoms.append(running)
            running += value
        heights.append(value)
        colours.append(colour)
        labels.append(wrap_label(label))
    exit_label = 'Exit\nequity' if row['is_realised'] else 'Current\nNAV'
    labels.append(exit_label)
    heights.append(running)
    bottoms.append(0.0)
    colours.append(COLOURS['anchor'])
    x = np.arange(len(labels))
    for i in range(len(labels)):
        h = heights[i]
        b = bottoms[i]
        ax.bar(
            x[i], abs(h) / 1e6,
            bottom=b / 1e6 if h >= 0 else (b + h) / 1e6,
            color=colours[i], width=0.6,
            edgecolor=None, linewidth=0,
        )
        ax.text(
            x[i], (b + h / 2) / 1e6,
            fmt_mm(h),
            ha='center', va='center',
            fontsize=7, color='white', fontweight='bold',
        )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_ylabel('EUR (MM)')
    ax.axhline(0, color='white', linewidth=0.5, linestyle='--')
    suffix = ' [REALISED]' if row['is_realised'] else ' [UNREALISED]'
    section_title(ax, f"{row['company_name']}{suffix}", fontsize=10)



In [ ]:
n         = len(rows)
fig, axes = plt.subplots(3, 2, figsize=(10, 10))
axes      = np.array(axes).flatten()

for i, row in enumerate(rows):
    waterfall_chart(row, axes[i])

for j in range(n, len(axes)):
    axes[j].set_visible(False)

sup_title(fig, 'Return Attribution — Value Bridge by Company', fontsize=13)
plt.tight_layout()
plt.subplots_adjust(hspace=0.4)

plt.subplots_adjust(top=.92)
save_fig(fig, FUND_ID, '06. value bridge by company')
plt.show()

In [ ]:
fig2, ax2 = plt.subplots(figsize=(10, 5))

av_fund      = ft['actual_value_created_eur']
gap_pct_fund = ft['reconciliation_gap_eur'] / av_fund if av_fund != 0 else float('nan')

waterfall_chart(
    {
        'company_name'           : 'Fund Total',
        'is_realised'            : all(r['is_realised'] for r in rows),
        'entry_equity_value'     : ft['total_cost_basis'],
        'ebitda_growth'          : ft['ebitda_growth_eur'],
        'multiple_expansion'     : ft['multiple_expansion_eur'],
        'leverage_effect'        : ft['leverage_effect_eur'],
        'distributions'          : ft['distributions_eur'],
        'reconciliation_gap'     : ft['reconciliation_gap_eur'],
        'reconciliation_gap_pct' : gap_pct_fund,
        'actual_value_created'   : av_fund,
        'gap_is_material'        : not np.isnan(gap_pct_fund) and abs(gap_pct_fund) > GAP_THRESHOLD,
    },
    ax2,
)

plt.tight_layout()
save_fig(fig2, FUND_ID, '06b. fund level value bridge')
plt.show()

print(f"\n{'Component':<25} {'EUR (MM)':>12} {'% of value created':>20}")
print('-' * 59)
for label, col in [
    ('EBITDA growth',      'ebitda_growth'),
    ('Multiple expansion', 'multiple_expansion'),
    ('Leverage effect',    'leverage_effect'),
    ('Distributions',      'distributions'),
    ('Reconciliation gap', 'reconciliation_gap'),
]:
    print(f"{label:<25} {ft[f'{col}_eur'] / 1e6:>11.1f}M {ft[f'{col}_pct']:>19.1%}")
print('-' * 59)
print(f"{'Actual value created':<25} {ft['actual_value_created_eur'] / 1e6:>11.1f}M")

In [ ]:
gap_rows = []
for r in rows:
    gap_rows.append({
        'Company':        r['company_name'],
        'Status':         'Realised' if r['is_realised'] else 'Unrealised',
        'Gap (EUR MM)':   f"{r['reconciliation_gap'] / 1e6:+.1f}M",
        'Gap (%)':        f"{r['reconciliation_gap_pct']:+.1%}",
        'Material':       'yes' if r['gap_is_material'] else '',
        'Note':           'Appraiser NAV includes DCF/growth premium outside EV/EBITDA bridge'
                          if r['gap_is_material'] and not r['is_realised'] else '',
    })

df_gaps = pd.DataFrame(gap_rows)
df_gaps.style.set_caption('Table 3.2 — Reconciliation Gap by Company')

---

## 6. Liquidity Profile

PE funds positions sit in the
**> 1 year** ESMA bucket. Liquidity risk in a PE fund comes entirely from
the **liability side**: unexpected capital calls, fund expenses, and the
inability to meet obligations if the fund has open-ended characteristics.

Under AIFMD Art. 16 and CSSF circular 18/698, the AIFM must monitor
liquidity risk and ensure the fund can meet its obligations. For a closed-end
PE fund this means maintaining sufficient cash or credit facilities to cover
unfunded commitments, management fees, and subscription line obligations.

<br>

#### A. Unfunded commitment

$\text{Unfunded} = \text{Committed capital} - \text{Drawn capital}$

<br>
<br>

#### B. Coverage ratio

$\text{Coverage Ratio} = \frac{\text{Cash} + \text{Sub-line headroom} + \text{Expected distributions (12m)}}{\text{Expected capital calls (12m)} + \text{Expected fees (12m)}}$

A coverage ratio below 1.0x signals a liquidity shortfall.

<br>
<br>

#### C. Capital call stress

Simulate an unexpected **30% drawdown** of remaining unfunded commitments.
This tests whether the fund can absorb an accelerated call schedule.

<br>
<br>

#### D. ESMA liquidity bucketing

ESMA guidelines (ESMA34-39-897) require liquidity stress testing for all AIFs,
including closed-end PE funds. For a PE fund **> 90% of assets should be in
the > 1 year bucket by construction**. The CSSF expects this table in the
risk report to confirm the fund is not misrepresenting its liquidity profile.
<br>
<br>

In [ ]:
import sqlalchemy as sa
from sqlalchemy.orm import Session

# ----------------------------------------------------------------
# 1. Committed vs drawn vs unfunded
# ----------------------------------------------------------------
with Session(ENGINE) as session:
    fund        = session.query(PEFund).filter_by(fund_id=FUND_ID).first()
    investments = session.query(PEFundInvestment).filter_by(fund_id=FUND_ID).all()

committed_eur = fund.target_size_eur

with ENGINE.connect() as conn:
    drawn = conn.execute(sa.text("""
        SELECT ABS(SUM(amount_eur)) FROM pe_cash_flows
        WHERE fund_id=:fid AND flow_type='capital_call'
    """), {'fid': FUND_ID}).scalar() or 0.0

    cm = conn.execute(sa.text("""
        SELECT cash_balance_eur, sub_line_drawn, sub_line_limit, net_cash_position
        FROM pe_fund_cash_management
        WHERE fund_id=:fid ORDER BY date DESC LIMIT 1
    """), {'fid': FUND_ID}).fetchone()

    dist_12m = conn.execute(sa.text("""
        SELECT SUM(amount_eur) FROM pe_cash_flows
        WHERE fund_id=:fid AND flow_type='distribution'
        AND date >= date(:vd, '-12 months') AND date <= :vd
    """), {'fid': FUND_ID, 'vd': VALUATION_DATE}).scalar() or 0.0

    fees_12m = conn.execute(sa.text("""
        SELECT ABS(SUM(amount_eur)) FROM pe_cash_flows
        WHERE fund_id=:fid AND flow_type='management_fee'
        AND date >= date(:vd, '-12 months') AND date <= :vd
    """), {'fid': FUND_ID, 'vd': VALUATION_DATE}).scalar() or 0.0

    calls_12m = conn.execute(sa.text("""
        SELECT ABS(SUM(amount_eur)) FROM pe_cash_flows
        WHERE fund_id=:fid AND flow_type='capital_call'
        AND date >= date(:vd, '-12 months') AND date <= :vd
    """), {'fid': FUND_ID, 'vd': VALUATION_DATE}).scalar() or 0.0

    nav_row = conn.execute(sa.text("""
        SELECT nav_eur FROM pe_nav_history
        WHERE fund_id=:fid AND company_id IS NULL AND date <= :vd
        ORDER BY date DESC LIMIT 1
    """), {'fid': FUND_ID, 'vd': VALUATION_DATE}).fetchone()

cash_balance      = cm[0]
sub_line_drawn    = cm[1]
sub_line_limit    = cm[2]
sub_line_headroom = sub_line_limit - sub_line_drawn
unfunded          = committed_eur - drawn
total_nav         = nav_row[0] if nav_row else 0.0
liquid_resources  = cash_balance + sub_line_headroom + dist_12m
obligations_12m   = calls_12m + fees_12m
coverage_ratio    = liquid_resources / obligations_12m if obligations_12m > 0 else float('inf')
stress_call       = unfunded * 0.30
stressed_resources= cash_balance + sub_line_headroom
shortfall         = max(0.0, stress_call - stressed_resources)

# ----------------------------------------------------------------
# 2. Summary table
# ----------------------------------------------------------------
summary = pd.DataFrame([
    ('Committed capital',          f'EUR {committed_eur/1e6:,.1f}M', ''),
    ('Drawn capital',              f'EUR {drawn/1e6:,.1f}M',         f'{drawn/committed_eur:.1%} of committed'),
    ('Unfunded commitments',       f'EUR {unfunded/1e6:,.1f}M',      f'{unfunded/committed_eur:.1%} of committed'),
    ('',                           '',                                ''),
    ('Cash balance',               f'EUR {cash_balance/1e6:,.1f}M',  ''),
    ('Sub-line drawn',             f'EUR {sub_line_drawn/1e6:,.1f}M',''),
    ('Sub-line limit',             f'EUR {sub_line_limit/1e6:,.1f}M',''),
    ('Sub-line headroom',          f'EUR {sub_line_headroom/1e6:,.1f}M', ''),
    ('',                           '',                                ''),
    ('Expected distributions 12m', f'EUR {dist_12m/1e6:,.1f}M',     ''),
    ('Expected capital calls 12m', f'EUR {calls_12m/1e6:,.1f}M',    'Investment period ended 2023-12-31'),
    ('Expected fees 12m',          f'EUR {fees_12m/1e6:,.1f}M',     ''),
    ('',                           '',                                ''),
    ('Coverage ratio',             f'{coverage_ratio:.2f}x',
     'PASS' if coverage_ratio >= 1.0 else 'FAIL ⚠'),
    ('Stress call (30% unfunded)', f'EUR {stress_call/1e6:,.1f}M',  ''),
    ('Stress shortfall',           f'EUR {shortfall/1e6:,.1f}M',
     'PASS' if shortfall == 0 else 'FAIL ⚠'),
], columns=['Metric', 'Value', 'Note'])

display(summary.style.set_caption('Table 6.1 — Liquidity Profile Summary').hide(axis='index'))

# ----------------------------------------------------------------
# 3. Unfunded per company
# ----------------------------------------------------------------
with ENGINE.connect() as conn:
    calls_per_co = conn.execute(sa.text("""
        SELECT company_id, ABS(SUM(amount_eur)) as drawn
        FROM pe_cash_flows
        WHERE fund_id=:fid AND flow_type='capital_call'
        GROUP BY company_id
    """), {'fid': FUND_ID}).fetchall()

calls_map    = {r[0]: r[1] for r in calls_per_co}
unfunded_rows = []
for inv in investments:
    drawn_co = calls_map.get(inv.company_id, 0.0)
    unf_co   = max(0.0, inv.cost_basis_eur - drawn_co)
    unfunded_rows.append({
        'Company':            inv.company_id,
        'Cost basis (EUR M)': f"{inv.cost_basis_eur/1e6:,.1f}",
        'Drawn (EUR M)':      f"{drawn_co/1e6:,.1f}",
        'Unfunded (EUR M)':   f"{unf_co/1e6:,.1f}",
        'Status':             'Exited' if inv.exit_date else 'Active',
    })

display(pd.DataFrame(unfunded_rows).style.set_caption(
    'Table 6.2 — Unfunded Commitments by Company').hide(axis='index'))

# ----------------------------------------------------------------
# 4. ESMA liquidity bucket table
# ----------------------------------------------------------------
illiquid_eur = total_nav - cash_balance
illiquid_pct = illiquid_eur / total_nav * 100 if total_nav else 0

buckets = pd.DataFrame([
    ('1 day',        'Cash',         f'EUR {cash_balance/1e6:,.1f}M',  f'{cash_balance/total_nav*100:.1f}%'),
    ('2-7 days',     '-',            'EUR 0.0M',                        '0.0%'),
    ('8-30 days',    '-',            'EUR 0.0M',                        '0.0%'),
    ('31-90 days',   '-',            'EUR 0.0M',                        '0.0%'),
    ('91-180 days',  '-',            'EUR 0.0M',                        '0.0%'),
    ('181-365 days', '-',            'EUR 0.0M',                        '0.0%'),
    ('> 1 year',     'PE portfolio', f'EUR {illiquid_eur/1e6:,.1f}M',  f'{illiquid_pct:.1f}%'),
], columns=['Bucket', 'Asset', 'EUR', '% NAV'])

esma_flag = 'PASS' if illiquid_pct >= 90 else 'REVIEW'
print(f"ESMA bucket check — > 1 year: {illiquid_pct:.1f}%  [{esma_flag}]")
print(f"Total NAV: EUR {total_nav/1e6:,.1f}M\n")
display(buckets.style.set_caption('Table 6.3 — ESMA Liquidity Buckets (ESMA34-39-897)').hide(axis='index'))

In [ ]:
# ----------------------------------------------------------------
# ESMA liquidity bucket table + chart (consistent with other funds)
# ----------------------------------------------------------------
bucket_order = ['1 day', '2-7 days', '8-30 days', '31-90 days', '91-365 days', '> 1 year']

bucket_data = {
    '1 day'       : cash_balance,
    '2-7 days'    : 0.0,
    '8-30 days'   : 0.0,
    '31-90 days'  : 0.0,
    '91-365 days' : 0.0,
    '> 1 year'    : illiquid_eur,
}

bucket_full = pd.DataFrame([
    {'liquidity_bucket': b, 'abs_exposure': bucket_data[b],
     'pct_nav_abs': bucket_data[b] / total_nav * 100 if total_nav else 0}
    for b in bucket_order
])

esma_flag = 'PASS' if illiquid_pct >= 90 else 'REVIEW'
print(f"ESMA bucket check — > 1 year: {illiquid_pct:.1f}%  [{esma_flag}]")
print(f"Total NAV: EUR {total_nav/1e6:,.1f}M\n")

print(f"{'Bucket':<15} {'Abs Exposure (EUR)':>20} {'% NAV':>8}")
print('-' * 47)
for _, row in bucket_full.iterrows():
    if row['abs_exposure'] > 0:
        print(f"{row['liquidity_bucket']:<15} {row['abs_exposure']:>20,.0f} "
              f"{row['pct_nav_abs']:>7.1f}%")
    else:
        print(f"{row['liquidity_bucket']:<15} {'—':>20} {'—':>8}")
print('-' * 47)
print(f"{'Total':<15} {total_nav:>20,.0f} {100.0:>7.1f}%")

# Chart
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(bucket_full['liquidity_bucket'],
              bucket_full['pct_nav_abs'],
              color=ACCENT, alpha=0.85, width=0.5)
ax.set_ylabel('Absolute Exposure (% NAV)', fontsize=9)
section_title(ax, "Liquidity Profile — Absolute Exposure by Bucket")

ax.tick_params(labelsize=9, length=0)
for bar, val in zip(bars, bucket_full['pct_nav_abs']):
    if val > 2:
        ax.text(bar.get_x() + bar.get_width()/2, val / 2,
                f'{val:.1f}%', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')
plt.tight_layout()
save_fig(fig, FUND_ID, "Liquidity buckets PE")
plt.show()



### 3b. PME Benchmark Comparison — Long-Nickels

The J-curve shows absolute performance. The Public Market Equivalent (PME)
answers the LP's most important question: **would I have done better in the
stock market?**

**Long-Nickels methodology** (industry standard, referenced in ILPA reporting
guidelines):

1. At each capital call date: invest the same amount in the index
2. At each distribution date: sell index units worth the distribution amount
3. At the valuation date: residual index portfolio = PME terminal NAV

$$PME\ IRR = \text{XIRR on replicated cash flows with PME terminal NAV}$$

$$\alpha = PE\ IRR - PME\ IRR$$

$\alpha > 0$: PE outperformed public markets — the illiquidity premium was
earned. $\alpha < 0$: public markets outperformed.

Benchmark: **SX5E (Euro Stoxx 50)** — EUR-denominated, appropriate for a
European buyout fund. Cash flows include capital calls, management fees,
distributions, and exit proceeds — the full LP net experience.


In [ ]:
from src.risk.pe_utils import pme_long_nickels

# ── 1. LP cash flows: all calls/fees out, all distributions/exits in ─────────
with ENGINE.connect() as conn:
    cf_raw = pd.read_sql(
        f"SELECT date, flow_type, amount_eur FROM pe_cash_flows "
        f"WHERE fund_id='{FUND_ID}' ORDER BY date", conn
    )
    nav_raw = pd.read_sql(
        f"SELECT date, nav_eur FROM pe_nav_history "
        f"WHERE fund_id='{FUND_ID}' AND company_id IS NULL ORDER BY date", conn
    )

cf_raw['date']  = pd.to_datetime(cf_raw['date'])
nav_raw['date'] = pd.to_datetime(nav_raw['date'])

lp_flows = cf_raw[cf_raw['flow_type'].isin([
    'capital_call', 'management_fee', 'distribution', 'exit_proceeds'
])].sort_values('date')

cfs_list   = lp_flows['amount_eur'].tolist()
dates_list = lp_flows['date'].tolist()
current_nav = nav_raw['nav_eur'].iloc[-1]

# ── 2. SX5E prices ────────────────────────────────────────────────────────────
sx5e_raw    = BBG.bdh('SX5E Index', 'PX_LAST', '20170101', '20260513')
sx5e_prices = sx5e_raw['PX_LAST'].copy()
sx5e_prices.index = pd.to_datetime(sx5e_prices.index)

# ── 3. Run PME ────────────────────────────────────────────────────────────────
pme = pme_long_nickels(
    cfs_list, dates_list, sx5e_prices,
    terminal_nav=current_nav, valuation_date=VALUATION_DATE
)

# ── 4. Summary table ──────────────────────────────────────────────────────────
flag = '✓ PE outperforms' if pme['alpha'] and pme['alpha'] > 0 else '⚠ Public markets outperform'
rows = [
    ('PE IRR (net, XIRR)',     f"{pme['pe_irr']:.2%}"          if pme['pe_irr']  else 'N/A', ''),
    ('PME IRR (SX5E)',         f"{pme['pme_irr']:.2%}"         if pme['pme_irr'] else 'N/A', ''),
    ('Alpha (PE – PME)',       f"{pme['alpha']:+.2%}"           if pme['alpha']   else 'N/A', flag),
    ('PME multiple (SX5E)',    f"{pme['pme_multiple']:.2f}x"   if pme['pme_multiple'] else 'N/A', ''),
    ('PME terminal NAV',       f"EUR {pme['pme_terminal_nav']/1e6:,.1f}M", 'Simulated index portfolio'),
    ('Actual PE NAV',          f"EUR {current_nav/1e6:,.1f}M",  'At valuation date'),
]
display(pd.DataFrame(rows, columns=['Metric', 'Value', 'Note'])
        .style.set_caption('Table 3b.1 — Long-Nickels PME (benchmark: SX5E)')
        .hide(axis='index'))

# ── 5. Chart ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 2))

labels = ['PE IRR\n(net)', 'PME IRR\n(SX5E)', 'Alpha\n(PE – PME)']
values = [pme['pe_irr'] * 100, pme['pme_irr'] * 100, pme['alpha'] * 100]
colors = [ACCENT, ACCENT2,
          '#27ae60' if pme['alpha'] > 0 else '#e74c3c']

bars = ax.bar(labels, values, color=colors, alpha=0.85, width=0.45)
ax.axhline(0, color='white', linewidth=0.8, linestyle='--')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height()/2 + (0.3 if val >= 0 else -0.6),
            f'{val:+.2f}%', ha='center', fontsize=8, fontweight='bold')

ax.set_ylabel('Annual return (%)')
section_title(ax, 'PE vs Public Market Equivalent — Long-Nickels (SX5E)')
plt.tight_layout()
save_fig(fig, FUND_ID, "PE vs Public Market Equivalent — Long-Nickels (SX5E)")
plt.show()


---
## 7. Annex VI Stress Testing — PE-Specific Scenarios

ESMA/2020/1498 (Annex VI) requires AIFMs to conduct regular stress tests calibrated to the fund strategy. For a PE buyout fund the relevant drivers are: **portfolio company earnings**, **exit multiples** (EV/EBITDA), and **sector concentration**. Liquidity stress for PE is covered in Section 6; this section focuses on NAV sensitivity.

Five scenarios are applied:

| # | Scenario | Shock |
|---|---|---|
| S1 | Uniform NAV markdown | −20% on all active company NAVs |
| S2 | Exit multiple compression | −2.0× EV/EBITDA on all active companies |
| S3 | Revenue / EBITDA stress | −15% revenue → −15% EBITDA → ΔNAVᵢ = −15% × EBITDAᵢ × multipleᵢ |
| S4 | Sector concentration — Technology | −30% NAV markdown on Technology holdings only |
| S5 | Historical — 2008 GFC proxy | −35% on all active company NAVs |

For multiples-based shocks the decomposition is: ΔNAV = ΔMultiple × EBITDA + ΔEBITDA × Multiple.

In [ ]:
# MRS-60: Fetch latest valuation data for active portfolio companies
from sqlalchemy import text as _text

with ENGINE.connect() as conn:
    _df_val = pd.read_sql(_text("""
        SELECT
            v.company_id,
            c.company_name,
            c.sector,
            v.date,
            v.appraised_nav_eur,
            v.ebitda_ltm_eur,
            v.revenue_ltm_eur,
            v.ev_ebitda,
            v.net_debt_eur
        FROM pe_valuation_report v
        JOIN pe_portfolio_companies c ON v.company_id = c.company_id
        WHERE c.status = 'Active'
        ORDER BY v.company_id, v.date DESC
    """), conn)

# Latest valuation per company (first row after ORDER BY date DESC)
_active = (
    _df_val
    .groupby('company_id', as_index=False)
    .first()
    .sort_values('company_id')
    .reset_index(drop=True)
)

_total_nav_active = _active['appraised_nav_eur'].sum()

print(f"Active portfolio companies: {len(_active)}")
print(f"Total active NAV: EUR {_total_nav_active/1e6:,.1f}M")
print()

_disp_cols = ['company_id', 'company_name', 'sector',
              'appraised_nav_eur', 'ebitda_ltm_eur', 'ev_ebitda']
_fmt = _active[_disp_cols].copy()
_fmt['appraised_nav_eur'] = _fmt['appraised_nav_eur'].apply(lambda x: f"EUR {x/1e6:,.1f}M")
_fmt['ebitda_ltm_eur']    = _fmt['ebitda_ltm_eur'].apply(
    lambda x: f"EUR {x/1e6:,.1f}M" if x > 0 else f"EUR ({abs(x)/1e6:,.1f}M)")
_fmt['ev_ebitda']         = _fmt['ev_ebitda'].apply(lambda x: f"{x:.1f}x")
_fmt.columns = ['ID', 'Company', 'Sector', 'NAV', 'EBITDA LTM', 'EV/EBITDA']
display(_fmt.style.set_caption('Table 7.1 — Active Portfolio — Latest Valuations')
        .hide(axis='index'))


In [ ]:
# MRS-60: Apply stress scenarios and build summary table

_results = []

for _, row in _active.iterrows():
    nav   = row['appraised_nav_eur']
    ebitda = row['ebitda_ltm_eur']
    mult   = row['ev_ebitda']
    sector = row['sector']

    # S1: Uniform NAV markdown -20%
    s1 = -0.20 * nav

    # S2: Multiple compression -2.0x  → ΔNAV = -2.0 × EBITDA
    s2 = -2.0 * ebitda

    # S3: Revenue/EBITDA stress -15%  → ΔNAV = -15% × EBITDA × multiple
    s3 = -0.15 * ebitda * mult

    # S4: Technology sector -30% (others: 0%)
    s4 = -0.30 * nav if sector == 'Technology' else 0.0

    # S5: Historical 2008 GFC proxy -35%
    s5 = -0.35 * nav

    _results.append({
        'company_id'  : row['company_id'],
        'company_name': row['company_name'],
        'sector'      : sector,
        'nav_eur'     : nav,
        's1_delta'    : s1,
        's2_delta'    : s2,
        's3_delta'    : s3,
        's4_delta'    : s4,
        's5_delta'    : s5,
    })

_stress = pd.DataFrame(_results)

# Fund-level totals
_totals = {
    'S1 — NAV markdown −20%'        : _stress['s1_delta'].sum(),
    'S2 — Multiple compression −2×'  : _stress['s2_delta'].sum(),
    'S3 — Revenue stress −15%'       : _stress['s3_delta'].sum(),
    'S4 — Technology concentration' : _stress['s4_delta'].sum(),
    'S5 — 2008 GFC proxy −35%'       : _stress['s5_delta'].sum(),
}

# Per-company display table
_disp = _stress[[
    'company_id', 'company_name', 'sector', 'nav_eur',
    's1_delta', 's2_delta', 's3_delta', 's4_delta', 's5_delta',
]].copy()

for col in ['nav_eur', 's1_delta', 's2_delta', 's3_delta', 's4_delta', 's5_delta']:
    _disp[col] = _disp[col].apply(lambda x: f"({abs(x)/1e6:,.1f})" if x < 0 else f"{x/1e6:,.1f}")

_disp.columns = ['ID', 'Company', 'Sector', 'NAV (EUR M)',
                 'S1 (EUR M)', 'S2 (EUR M)', 'S3 (EUR M)',
                 'S4 (EUR M)', 'S5 (EUR M)']
display(_disp.style.set_caption('Table 7.2 — NAV Impact by Scenario and Company (EUR M, losses in brackets)')
        .hide(axis='index'))

# Fund-level summary
print("\nTable 7.3 — Fund-Level Stress Summary")
print(f"{'':<38} {'ΔNAV (EUR M)':>14} {'ΔNAV (% NAV)':>14}")
print('-' * 68)
for label, delta in _totals.items():
    pct = delta / _total_nav_active * 100
    sign = '' if delta >= 0 else ''
    print(f"{label:<38} {delta/1e6:>13,.1f}M {pct:>13.1f}%")
print('-' * 68)
print(f"Base NAV (active companies)          {_total_nav_active/1e6:>13,.1f}M")


In [ ]:
_labels = ['S1\n-20% NAV', 'S2\n-2× multiple', 'S3\n-15% revenue',
           'S4\nTech -30%', 'S5\nGFC -35%']
_deltas_m = [v / 1e6 for v in _totals.values()]
_pcts     = [v / _total_nav_active * 100 for v in _totals.values()]
_base_m   = _total_nav_active / 1e6

fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(_labels, _deltas_m, color=C['red'], alpha=0.85, width=0.5)
ax.axhline(0, color='grey', lw=0.8)
ax.set_ylabel('ΔNAV (EUR M)')
ax.set_ylim(-300)


for bar, val, pct in zip(bars, _deltas_m, _pcts):
    cx = bar.get_x() + bar.get_width() / 2
    # percentage inside the bar
    ax.text(cx, val / 2,
            f'{pct:.1f}%', ha='center', va='center',
            fontsize=10, color='white', fontweight='bold')
    # absolute value below the bar
    ax.text(cx, val - 10,
            f'{val:,.1f}M', ha='center', va='top',
            fontsize=9, color='white')
    

sup_title(fig,
    f'Annex VI Stress Testing — PE Buyout Fund\n'
    f'Base NAV (active): EUR {_base_m:,.1f}M  |  Valuation date: {VALUATION_DATE}',
    fontsize=10)

plt.tight_layout()
save_fig(fig, FUND_ID, '10. stress test annex vi PE')
plt.show()

print("\nAnnex VI Stress Testing — regulatory note:")
print("Scenarios align with ESMA/2020/1498 Annex VI requirements for PE/closed-end AIFs.")
print("S2 and S3 use the decomposition ΔNAV = ΔMultiple × EBITDA + ΔEBITDA × Multiple.")
print("S4 isolates sector concentration risk — Technology = PE_001 + PE_008.")
print("Results reported to the risk committee quarterly and included in CSSF Annex IV filing.")

---

## 7. Annex IV Report — MRS-59

PE buyout funds report under AIFMD Annex IV with specific fields for closed-ended
private equity vehicles. Key differences from liquid fund reporting:

- **Leverage**: only fund-level borrowing (subscription credit line) enters the AIFMD
  leverage calculation. Portfolio company debt is ring-fenced at SPV level and excluded
  per EU231/2013 Art. 7 project finance treatment.
- **Exposures**: reported by sector, country, and investment stage (early/growth/buyout)
  rather than daily market prices — PE positions have no daily liquidity.
- **Unfunded commitments**: disclosed as contingent leverage — the CSSF monitors whether
  remaining commitments can be funded from undrawn capital and the subscription line.

AIFMD II (Directive 2024/927/EU) added mandatory disclosure of available liquidity
management tools (LMTs), delegation arrangements, and expanded principal market fields.

**Regulatory basis:** AIFMD Art. 110 / EU231/2013 Annex IV / ESMA v1.7 (Jul 2024) / AIFMD II


In [ ]:
from src.reporting.annex_iv import build_annex_iv, export_annex_iv_excel

PE_FUND_ID       = 'AIFM_PE_Buyout'
ANNEX_IV_QUARTER = '2026-03-31'

annex_iv = build_annex_iv(ENGINE, PE_FUND_ID, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV — {PE_FUND_ID}   Reporting period: Q1 2026 ({ANNEX_IV_QUARTER})")
print(f"Fund NAV: EUR {annex_iv['_nav']/1e6:,.1f}M")

print("\n=== Section 1 — Fund Identification ===")
display(annex_iv['identification'])


In [ ]:
# Section 2 — Exposures (cost basis, not market price)
print("\n=== Section 2.1 — Portfolio by Sector ===")
display(annex_iv['sector_exposure'])

print("\n=== Section 2.2 — Portfolio by Country ===")
display(annex_iv['country_exposure'])

print("\n=== Section 2.3 — Portfolio by Investment Stage ===")
display(annex_iv['stage_exposure'])

print("\n=== Section 2.4 — Top 5 Investments ===")
display(annex_iv['top5_positions'])


In [ ]:
# Section 3 — Leverage (fund level only; project debt disclosed separately)
print("\n=== Section 3 — Leverage ===")
display(annex_iv['leverage_detail'])

# Section 5 — Performance (IRR, DPI, TVPI, fund life)
print("\n=== Section 5 — Fund Performance & Capital Account ===")
display(annex_iv['performance'])

# AIFMD II expanded disclosures
print("\n=== AIFMD II — LMTs, Delegation, Unfunded Commitments ===")
display(annex_iv['aifmd_ii_disclosure'])


In [ ]:
# Export full Annex IV workbook (all five funds)
path = export_annex_iv_excel(ENGINE, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV workbook written: {path}")


## ESG Risk IndicatorsThe notebook builds PE ESG indicators using `build_private_esg_df`. The output uses the same column structure as the listed-asset ESG workflow, while adding `esg_reporter` to identify the data source.The output summarises portfolio-company ESG indicators and controversy flags.

In [ ]:
from src.risk.esg_utils import build_private_esg_df, esg_portfolio_summary, ESG_THRESHOLD_LOW

CURRENT_QUARTER = '2026-03-31'   # last quarter-end before VALUATION_DATE

esg_df  = build_private_esg_df(FUND_ID, CURRENT_QUARTER, 'pe', ENGINE)
NAV_ESG = esg_df['esg_exposure_eur'].sum()
summary = esg_portfolio_summary(esg_df, NAV_ESG)

print("ESG DATA QUALITY")
print(f"  Reporters   : {esg_df['esg_reporter'].value_counts().to_dict()}")
print(f"  Report date : {esg_df['esg_report_date'].unique().tolist()}")
print()
print("PORTFOLIO ESG SUMMARY")
print(f"  Weighted avg ESG score  : {summary['wav_esg']:.1f}")
print(f"  Weighted avg ENV score  : {summary['wav_env']:.1f}")
print(f"  Weighted avg SOC score  : {summary['wav_soc']:.1f}")
print(f"  Weighted avg GOV score  : {summary['wav_gov']:.1f}")
print(f"  Avg carbon intensity    : {summary['wav_carbon']:.1f} tCO2e / EUR M")
print(f"  % below ESG threshold   : {summary['pct_low_esg']:.1f}%  (threshold = {ESG_THRESHOLD_LOW})")
print(f"  % controversy exposure  : {summary['pct_controversy']:.1f}%")
print()

esg_df[['instrument_name', 'sub_asset_class', 'esg_score', 'env_score',
        'soc_score', 'gov_score', 'controversy_flag',
        'carbon_intensity', 'esg_reporter', 'weight_pct']].style \
    .background_gradient(subset=['esg_score'], cmap='RdYlGn', vmin=0, vmax=100) \
    .format({'esg_score': '{:.0f}', 'env_score': '{:.0f}', 'soc_score': '{:.0f}',
             'gov_score': '{:.0f}', 'carbon_intensity': '{:.1f}', 'weight_pct': '{:.1f}%'})
